In [1]:
import sqlite3
import litellm

# 1. Create an in-memory relational database connection
db_connection = sqlite3.connect(":memory:")
cursor = db_connection.cursor()

# 2. Build a structured table for tracking system incidents
cursor.execute("""
CREATE TABLE system_alerts (
    alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
    cluster_name TEXT,
    port_number INTEGER,
    alert_type TEXT,
    severity TEXT,
    incident_date TEXT
)
""")

# 3. Insert structured log records
mock_alerts = [
    ("MeshQuery", 9092, "Heartbeat_Failure", "Critical", "2026-06-01"),
    ("MeshQuery", 9092, "Port_Unreachable", "Critical", "2026-06-01"),
    ("Payment_DB", 5432, "OutOfMemory", "Urgent", "2026-06-01"),
    ("MeshQuery", 9092, "High_Latency_RSocket", "Warning", "2026-06-02"),
    ("Auth_Service", 443, "Expired_SSL_Certificate", "Urgent", "2026-06-02")
]

cursor.executemany("""
INSERT INTO system_alerts (cluster_name, port_number, alert_type, severity, incident_date)
VALUES (?, ?, ?, ?, ?)
""", mock_alerts)

db_connection.commit()
print("✓ Cell 1: Relational SQLite database initialized and seeded with 5 operational log rows.")

18:52:27 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
18:52:27 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


✓ Cell 1: Relational SQLite database initialized and seeded with 5 operational log rows.


In [2]:
def generate_sql_query(user_question: str) -> str:
    # We define the database schema explicitly as a text string to give the LLM full structural awareness
    database_schema_blueprint = """
    Table: system_alerts
    Columns:
    - alert_id (INTEGER)
    - cluster_name (TEXT) -> Examples: 'MeshQuery', 'Payment_DB', 'Auth_Service'
    - port_number (INTEGER)
    - alert_type (TEXT)
    - severity (TEXT) -> Examples: 'Critical', 'Urgent', 'Warning'
    - incident_date (TEXT) -> Format: 'YYYY-MM-DD'
    """
    
    prompt = [
        {
            "role": "system",
            "content": (
                "You are an expert database administrator. Your task is to convert a user's natural language question "
                "into a single valid SQL query matching the SQLite database schema provided below.\n\n"
                f"Database Schema:\n{database_schema_blueprint}\n"
                "Rules:\n"
                "1. Return ONLY the raw executable SQL query string.\n"
                "2. Do not include markdown formatting (like ```sql), code blocks, or explanation text.\n"
                "3. Ensure string comparisons match the casing shown in the examples."
            )
        },
        {
            "role": "user",
            "content": f"Question: {user_question}"
        }
    ]
    
    response = litellm.completion(
        model="ollama/llama3.2",
        messages=prompt,
        api_base="http://localhost:11434",
        temperature=0.0 # Strict determinism
    )
    
    raw_sql = response.choices[0].message.content.strip()
    
    # Standard cleanup safety handle to strip accidental markdown wrapper blocks
    if raw_sql.startswith("```sql"):
        raw_sql = raw_sql.replace("```sql", "").replace("```", "").strip()
    elif raw_sql.startswith("```"):
        raw_sql = raw_sql.replace("```", "").strip()
        
    return raw_sql

print("✓ Cell 2: SQL generation translator successfully compiled.")

✓ Cell 2: SQL generation translator successfully compiled.


In [5]:
def execute_text_to_sql_rag(user_prompt: str):
    print(f"\nUser Query: '{user_prompt}'")
    
    # Step A: Generate the raw SQL command string
    generated_sql = generate_sql_query(user_prompt)
    print(f"Generated SQL: Code -> [ {generated_sql} ]")
    
    # Step B: Run the query string directly against our SQLite database engine
    try:
        cursor.execute(generated_sql)
        query_results = cursor.fetchall()
        print(f"Database Result Matrix: {query_results}")
    except Exception as err:
        print(f"❌ Database execution failed: {err}")
        return
    
    # Step C: Synthesize the raw relational dataset rows back into natural language
    synthesis_messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful operations engineer. Your job is to take the raw structured SQL database query results "
                "and explain the answer clearly to the user in a short sentence based on those numbers."
            )
        },
        {
            "role": "user",
            "content": f"Question: {user_prompt}\nRaw DB Data Output: {query_results}"
        }
    ]
    
    print("AI Answer: ", end="", flush=True)
    response = litellm.completion(
        model="ollama/llama3.2",
        messages=synthesis_messages,
        api_base="http://localhost:11434",
        stream=True
    )
    
    for chunk in response:
        token = chunk.choices[0].delta.content or ""
        print(token, end="", flush=True)
    print("\n" + "="*60)

print("✓ Cell 3: Text-to-SQL orchestration layer bound.")

✓ Cell 3: Text-to-SQL orchestration layer bound.


In [6]:
# Test Case 1: Filtering and Aggregation Counting
execute_text_to_sql_rag("How many critical alerts were thrown by the MeshQuery cluster?")

# Test Case 2: Specific Date Filtering
execute_text_to_sql_rag("List the alert types that occurred specifically on 2026-06-02.")


User Query: 'How many critical alerts were thrown by the MeshQuery cluster?'
Generated SQL: Code -> [ SELECT COUNT(alert_id) FROM system_alerts WHERE severity = 'Critical' AND cluster_name = 'MeshQuery'; ]
Database Result Matrix: [(2,)]
AI Answer: There was 1 critical alert thrown by the MeshQuery cluster.

User Query: 'List the alert types that occurred specifically on 2026-06-02.'
Generated SQL: Code -> [ SELECT alert_type FROM system_alerts WHERE incident_date = '2026-06-02' ]
Database Result Matrix: [('High_Latency_RSocket',), ('Expired_SSL_Certificate',)]
AI Answer: The 'Expired_SSL_Certificate' alert type occurred on June 2nd, 2026.
